In [0]:
%pip install azure-eventhub

In [0]:
 dbutils.library.restartPython()

In [0]:
import json
import random
from datetime import datetime, timedelta
from azure.eventhub import EventHubProducerClient, EventData

# Initial state
last_close = 200.0
current_date = datetime(2026, 4, 1)

producer = EventHubProducerClient.from_connection_string(conn_str)
batch = producer.create_batch()

def generate_next(last_close, current_date):
    open_price = last_close + random.uniform(-1, 1)
    close_price = open_price + random.uniform(-2, 2)

    high_price = max(open_price, close_price) + random.uniform(0, 2)
    low_price = min(open_price, close_price) - random.uniform(0, 2)

    data = {
        "open": round(open_price, 2),
        "close": round(close_price, 2),
        "low": round(low_price, 2),
        "high": round(high_price, 2),
        "sector": "Banking",
        "timestamp": current_date.isoformat()
    }

    return data, close_price, current_date + timedelta(days=1)

# 🔁 Generate 20 days = 20 events
for _ in range(40):
    record, last_close, current_date = generate_next(last_close, current_date)
    batch.add(EventData(json.dumps(record)))

# Send batch
producer.send_batch(batch)
producer.close()